# Flight Price Prediction — Databricks Notebook
**Dataset:** Expedia Itineraries (~82M rows) from Azure Blob Storage  
**Goal:** Feature Engineering → EDA → Linear Regression modelling

## 0. Mount Azure Blob Storage

In [ ]:
storage_account = "mydatastoarege1"
access_key = "****"  # Replace with your actual access key or use dbutils.secrets.get()
container = "myflightdata"
file_name = "itineraries.csv"


## 1. Load Dataset

In [ ]:


# Construct the wasbs:// path with embedded credentials (works on serverless)
wasbs_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/{file_name}"

# Read the file using the wasbs:// path with credentials
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("fs.azure.account.key." + storage_account + ".blob.core.windows.net", access_key) \
    .load(wasbs_path)

display(df)


In [ ]:
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

## 2. Preliminary Analysis — Select Relevant Columns


- Core Columns:  flightDate, startingAirport, destinationAirport, travelDuration, elapsedDays, isBasicEconomy, isRefundable, isNonStop, totalFare, seatsRemaining, totalTravelDistance, segmentsEquipmentDescription and segmentsAirlineName 

- Derived Tax Data: Create a taxes variable by subtracting the base fare from the total fare.

- Booking Lead Time: By transforming searchDate and flightDate into a daysFrom column, we can measure the "days until departure," a critical factor in predictive pricing models.

In [ ]:
relevant_cols = [
    'flightDate', 'searchDate', 'startingAirport', 'destinationAirport',
    'travelDuration', 'elapsedDays', 'isBasicEconomy', 'isRefundable',
    'isNonStop', 'baseFare', 'totalFare', 'seatsRemaining',
    'totalTravelDistance', 'segmentsAirlineName', 'segmentsEquipmentDescription'
]

data = df.select(relevant_cols)
display(data.limit(5))

## 3. Feature Engineering

In [ ]:
# Create taxes column: totalFare - baseFare
data = data.withColumn('taxes', F.col('totalFare') - F.col('baseFare'))

# Parse dates and create daysFrom column
data = data \
    .withColumn('flightDate', F.to_date('flightDate')) \
    .withColumn('searchDate', F.to_date('searchDate')) \
    .withColumn('daysFrom', F.datediff(F.col('flightDate'), F.col('searchDate')))

# Extract flight month and year for seasonal EDA
data = data \
    .withColumn('flightMonth', F.date_format('flightDate', 'MMMM')) \
    .withColumn('flightYear', F.year('flightDate'))

# Convert travelDuration (ISO 8601 e.g. PT2H30M) to hours
data = data.withColumn(
    'travelDuration',
    (
        F.coalesce(F.regexp_extract('travelDuration', r'(\d+)H', 1).cast('double'), F.lit(0)) +
        F.coalesce(F.regexp_extract('travelDuration', r'(\d+)M', 1).cast('double'), F.lit(0)) / 60
    ).cast('double')
)

# Convert booleans to integers
for col_name in ['isBasicEconomy', 'isRefundable', 'isNonStop']:
    data = data.withColumn(col_name, F.col(col_name).cast('int'))

display(data.limit(5))

## 4. Handle Duplicates

In [ ]:
before = data.count()
data = data.dropDuplicates()
after = data.count()
print(f"Rows before: {before:,}")
print(f"Rows after dropping duplicates: {after:,}")
print(f"Duplicates removed: {before - after:,}")

## 5. Handle Null Values

In [ ]:
# Check null counts per column
null_counts = data.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in data.columns
])
display(null_counts)

In [ ]:
# Impute totalTravelDistance using median per (startingAirport, destinationAirport)
from pyspark.sql.window import Window

window_spec = Window.partitionBy('startingAirport', 'destinationAirport')

data = data.withColumn(
    'distGroup',
    F.when(
        F.col('totalTravelDistance').isNull(),
        F.percentile_approx('totalTravelDistance', 0.5).over(window_spec)
    ).otherwise(F.col('totalTravelDistance'))
)

# Replace nulls in segmentsEquipmentDescription with 'Missing'
data = data.withColumn(
    'segmentsEquipmentDescription',
    F.when(F.col('segmentsEquipmentDescription').isNull(), 'Missing')
     .otherwise(F.col('segmentsEquipmentDescription'))
)

# Verify nulls resolved
null_counts_after = data.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ['distGroup', 'segmentsEquipmentDescription']
])
display(null_counts_after)

## 6. Save as Parquet (for faster future reads)

In [ ]:
parquet_path = f"{mount_point}/itineraries.parquet"
data.write.mode('overwrite').parquet(parquet_path)
print(f"Saved to {parquet_path}")

## 7. Exploratory Data Analysis (EDA)
> EDA uses a sampled Pandas DataFrame for plotting. Adjust `fraction` as needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sample ~1M rows for EDA visualisations
sample_df = data.sample(fraction=0.012, seed=42).toPandas()
print(f"Sample size: {len(sample_df):,} rows")

In [ ]:
# Flights by Month
sns.countplot(data=sample_df, x='flightMonth',
              order=['January','February','March','April','May','June',
                     'July','August','September','October','November','December'])
plt.xticks(rotation=45)
plt.title('Number of Flights by Month')
plt.tight_layout()
plt.show()

In [ ]:
# Busiest Airports
airportCount = (
    sample_df['startingAirport'].value_counts() +
    sample_df['destinationAirport'].value_counts()
).reset_index()
airportCount.columns = ['airport', 'count']

plt.barh(airportCount['airport'], airportCount['count'])
plt.axvline(np.mean(airportCount['count']), color='red', label='Mean')
plt.title('Busiest US Airports')
plt.xlabel('Count')
plt.ylabel('Airport')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Most Frequent Airport Combinations
combo = sample_df.groupby(['startingAirport','destinationAirport']).size().nlargest(15).reset_index(name='count')
combo['Airport'] = combo['startingAirport'] + '-' + combo['destinationAirport']
plt.barh(combo['Airport'], combo['count'], log=True)
plt.xlabel('Count (log scale)')
plt.ylabel('Route')
plt.title('Top 15 Most Frequent Routes')
plt.tight_layout()
plt.show()

In [ ]:
# Most Popular Airlines
airlines = sample_df['segmentsAirlineName'].str.split('||').explode().value_counts()
airlines_plt = airlines[:6].copy()
airlines_plt['Other'] = airlines[6:].sum()
plt.pie(airlines_plt, labels=airlines_plt.index, autopct="%1.1f%%",
        wedgeprops={'linewidth': 0.5, 'edgecolor': 'black'})
plt.title('Most Popular Airlines')
plt.show()

In [ ]:
# Seasonal Fare Trends
month_order = ['April','May','June','July','August','September','October']
monthlyCost = sample_df.groupby('flightMonth')['totalFare'].mean()
monthlyCost = monthlyCost.reindex([m for m in month_order if m in monthlyCost.index])
plt.barh(monthlyCost.index, monthlyCost, log=True)
plt.title('Average Fare by Month (Seasonal Trends)')
plt.xlabel('Average Total Fare (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
# Mean Travel Duration by Airport
airport_grps = sample_df.groupby('startingAirport')['travelDuration'].mean()
plt.barh(airport_grps.index, airport_grps)
plt.axvline(airport_grps.mean(), color='r', label='Mean')
plt.ylabel('Airport')
plt.xlabel('Mean Duration (hours)')
plt.title('Mean Travel Duration by Takeoff Airport')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fare by Seat Type
sns.boxplot(x='isBasicEconomy', y='totalFare', hue='isBasicEconomy', data=sample_df)
plt.xlabel('Is Basic Economy (0=No, 1=Yes)')
plt.ylabel('Total Fare')
plt.title('Fare by Seat Type')
plt.show()

In [ ]:
# Correlation Heatmap
numeric_col = sample_df.select_dtypes(include='number')
sns.heatmap(numeric_col.corr(method='spearman'), cmap='plasma', annot=True)
plt.title('Spearman Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Linear Regression Modelling
> We use a sampled Pandas DataFrame for sklearn modelling. For production-scale, use MLlib.

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Use sample_df for modelling; drop remaining nulls in distGroup
model_df = sample_df.dropna(subset=['distGroup', 'travelDuration', 'totalFare', 'daysFrom'])
print(f"Modelling dataset size: {len(model_df):,}")

### 8a. Simple Linear Regression — Distance Only

In [ ]:
X_train1, X_test1, Y_train1, Y_test1 = train_test_split(
    model_df['distGroup'], model_df['totalFare'], test_size=0.05, random_state=42
)

ct1 = ColumnTransformer(transformers=[('numeric', StandardScaler(), ['distGroup'])])
pipeline1 = Pipeline(steps=[('preprocess', ct1), ('regressor', LinearRegression())])
pipeline1.fit(X_train1.to_frame(), Y_train1)
Y_pred1 = pipeline1.predict(X_test1.to_frame())

print(f"MAE:  {mean_absolute_error(Y_test1, Y_pred1):.4f}")
print(f"R2:   {r2_score(Y_test1, Y_pred1):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test1, Y_pred1):.4f}")

plt.scatter(X_test1, Y_test1, alpha=0.1, s=1)
plt.plot(sorted(X_test1), pipeline1.predict(pd.DataFrame(sorted(X_test1), columns=['distGroup'])), 'r')
plt.xlabel('Distance')
plt.ylabel('Total Fare')
plt.title('Simple Linear Regression — Distance vs Fare')
plt.show()

### 8b. Multiple Linear Regression — Distance + Duration

In [ ]:
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    model_df[['distGroup', 'travelDuration']], model_df['totalFare'], test_size=0.05, random_state=42
)

ct2 = ColumnTransformer(transformers=[('numeric', StandardScaler(), ['distGroup', 'travelDuration'])])
pipeline2 = Pipeline(steps=[('preprocess', ct2), ('regressor', LinearRegression())])
pipeline2.fit(X_train2, Y_train2)
Y_pred2 = pipeline2.predict(X_test2)

print(f"MAE:  {mean_absolute_error(Y_test2, Y_pred2):.4f}")
print(f"R2:   {r2_score(Y_test2, Y_pred2):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test2, Y_pred2):.4f}")

### 8c. Multiple Linear Regression — With Association (Interaction) Term

In [ ]:
preprocess3 = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(interaction_only=True, include_bias=False))
])
ct3 = ColumnTransformer(transformers=[('numeric', preprocess3, ['distGroup', 'travelDuration'])])
pipeline3 = Pipeline(steps=[('preprocess', ct3), ('regressor', LinearRegression())])
pipeline3.fit(X_train2, Y_train2)
Y_pred3 = pipeline3.predict(X_test2)

print(f"MAE:  {mean_absolute_error(Y_test2, Y_pred3):.4f}")
print(f"R2:   {r2_score(Y_test2, Y_pred3):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test2, Y_pred3):.4f}")

### 8d. Multiple Linear Regression — With Categorical Features

In [ ]:
data_mlr = model_df[[
    'startingAirport', 'destinationAirport', 'isBasicEconomy', 'isNonStop',
    'travelDuration', 'distGroup', 'segmentsAirlineName', 'flightMonth', 'totalFare', 'daysFrom'
]].copy()

# Use only the first airline segment
data_mlr['segmentsAirlineName'] = data_mlr['segmentsAirlineName'].str.split('||').str[0]
data_mlr = data_mlr.dropna()

X_train4, X_test4, Y_train4, Y_test4 = train_test_split(
    data_mlr.drop('totalFare', axis=1), data_mlr['totalFare'], test_size=0.05, random_state=42
)

ct4 = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), ['distGroup', 'travelDuration', 'daysFrom']),
    ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'),
     ['startingAirport', 'destinationAirport', 'isBasicEconomy', 'isNonStop', 'segmentsAirlineName', 'flightMonth'])
])

pipeline4 = Pipeline(steps=[('preprocess', ct4), ('regressor', LinearRegression())])
pipeline4.fit(X_train4, Y_train4)
Y_pred4 = pipeline4.predict(X_test4)

print(f"MAE:  {mean_absolute_error(Y_test4, Y_pred4):.4f}")
print(f"R2:   {r2_score(Y_test4, Y_pred4):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test4, Y_pred4):.4f}")

In [ ]:
# Residual Plot — check for heteroscedasticity
residuals = Y_test4 - Y_pred4
plt.scatter(Y_pred4, residuals, alpha=0.2, s=1)
plt.axhline(0, color='r')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot — Model 4')
plt.show()

### 8e. Log-Transformed Y — Addressing Heteroscedasticity

In [ ]:
Y_train_log = np.log1p(Y_train4)
Y_test_log = np.log1p(Y_test4)

pipeline4.fit(X_train4, Y_train_log)
Y_log_pred = pipeline4.predict(X_test4)

print(f"MAE:  {mean_absolute_error(Y_test_log, Y_log_pred):.4f}")
print(f"R2:   {r2_score(Y_test_log, Y_log_pred):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test_log, Y_log_pred):.4f}")

resid_log = Y_test_log - Y_log_pred
plt.scatter(Y_test_log, resid_log, alpha=0.2, s=1)
plt.axhline(0, color='r')
plt.title('Residual Plot — Log-Transformed Y')
plt.xlabel('Log(True Fare)')
plt.ylabel('Residuals')
plt.show()

### 8f. Best Model — Categorical + Association Terms

In [ ]:
pipeline_categorical = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore')),
    ('interaction', PolynomialFeatures(interaction_only=True, include_bias=False))
])

ct5 = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), ['distGroup', 'travelDuration', 'daysFrom']),
    ('categoric_interaction', pipeline_categorical, ['startingAirport', 'destinationAirport', 'flightMonth']),
    ('categoric', OneHotEncoder(drop='first', handle_unknown='ignore'), ['isBasicEconomy', 'isNonStop', 'segmentsAirlineName'])
])

pipeline5 = Pipeline(steps=[('preprocess', ct5), ('regressor', LinearRegression())])
pipeline5.fit(X_train4, Y_train_log)
Y_preds = pipeline5.predict(X_test4)

print(f"MAE:  {mean_absolute_error(Y_test_log, Y_preds):.4f}")
print(f"R2:   {r2_score(Y_test_log, Y_preds):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(Y_test_log, Y_preds):.4f}")

## 9. Conclusion

We evaluated multiple linear regression models to predict flight prices:

| Model | R² Score |
|---|---|
| Simple LR (distance only) | ~0.23 |
| Multiple LR (distance + duration) | ~0.25 |
| With interaction term | ~0.25 |
| With categorical features | ~0.49 |
| Log-transformed Y | ~0.60 |
| Categorical + association terms | ~0.63 |

**Key findings:**
- Route and seasonality are the strongest price predictors.
- Log-transforming Y addresses heteroscedasticity significantly.
- Linear models cannot fully capture airline pricing non-linearity — tree-based models (e.g. XGBoost) are recommended next steps.